# Session 13 — Tools & Function Calling

**The concept:** tool use is the bridge from chat to software. The model
never runs your code — it emits a structured request, your code executes
it, and you return a result.

**The lesson:** *a failing tool must return an error, not take down the agent.*

**No API key required.** We use the deterministic mock LLM.


In [ ]:
import json, os, sys, time, pathlib
sys.path.insert(0, os.path.join('..', 'src'))
print('python', sys.version.split()[0])


## 1. A tool is a JSON schema plus a function


In [ ]:
import tools as T

for s in T.registry.schemas():
    fn = s['function']
    print(f"{fn['name']:16s} -> {fn['description']}")


## 2. The dispatcher never raises

This is the whole point of the module. A tool can fail. The network can be
down. The arguments can be garbage. **None of that may crash the agent.**


In [ ]:
ok = T.dispatch('get_weather', {'city': 'Stockton, CA', 'units': 'f'})
print('success ->', ok.ok, ok.content)

bad = T.dispatch('flaky_database', {'mode': 'crash'})
print('failure ->', bad.ok, '|', bad.error)

unknown = T.dispatch('no_such_tool', {'x': 1})
print('unknown ->', unknown.ok, '|', unknown.error)


## 3. Schema validation catches bad arguments

A model that sends `units="kelvin"` to a tool that only understands
`c`/`f` should get a clean error — not a silently wrong temperature.


In [ ]:
missing = T.dispatch('get_weather', {})
print('missing arg  ->', missing.ok, '|', missing.error)

bad_enum = T.dispatch('get_weather', {'city': 'Stockton', 'units': 'kelvin'})
print('bad enum     ->', bad_enum.ok, '|', bad_enum.error)


## 4. Your turn — add a tool


In [ ]:
def get_inventory(sku: str) -> dict:
    """Look up stock on hand for a SKU."""
    return {'sku': sku, 'on_hand': 42, 'warehouse': 'Lathrop'}

T.registry.register(
    'get_inventory',
    'Look up stock on hand for a product SKU.',
    {'type': 'object',
     'properties': {'sku': {'type': 'string', 'description': 'Product SKU'}},
     'required': ['sku']},
    get_inventory,
)

print(T.dispatch('get_inventory', {'sku': 'CC3PO-001'}))


## Takeaway

- A tool is a **JSON schema** (for the model) plus a **function** (for you).
- The model proposes; your code disposes.
- **The dispatcher returns errors. It never raises.** That is the design.
- Narrow tools, explicit descriptions, typed returns, deterministic fallbacks.
